In [ ]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import tensorflow as tf
from google.colab import files
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import LinearSVC
from sklearn.metrics import accuracy_score, classification_report
from sklearn.metrics import accuracy_score
from sklearn.cluster import MiniBatchKMeans
from sklearn.metrics import adjusted_rand_score
from sklearn.decomposition import PCA

print("UPLOAD TRAIN CSV:")
uploaded = files.upload()
train_filename = list(uploaded.keys())[0]
train_df = pd.read_csv(train_filename, header=None)

print("\nUPLOAD TEST CSV:")
uploaded = files.upload()
test_filename = list(uploaded.keys())[0]
test_df = pd.read_csv(test_filename, header=None)

UPLOAD TRAIN CSV:


Saving emnist-letters-train.csv to emnist-letters-train (1).csv

UPLOAD TEST CSV:


Saving emnist-letters-test.csv to emnist-letters-test (1).csv


In [ ]:
y_train_full = train_df.iloc[:, 0].values
X_train_full = train_df.iloc[:, 1:].values

y_test_full = test_df.iloc[:, 0].values
X_test_full = test_df.iloc[:, 1:].values

number_of_classes = 37

X_train_full = X_train_full.astype("float32") / 255.0
X_test_full  = X_test_full.astype("float32") / 255.0

train_images_number = X_train_full.shape[0]
test_images_number  = X_test_full.shape[0]

X_train_full = X_train_full.reshape(train_images_number, 28, 28, 1)
X_test_full  = X_test_full.reshape(test_images_number, 28, 28, 1)

print("Train images:", X_train_full.shape, "Train labels:", y_train_full.shape)
print("Test images :", X_test_full.shape,  "Test labels :", y_test_full.shape)

X_train_split, X_val_split, y_train_split, y_val_split = train_test_split(
    X_train_full,
    y_train_full,
    test_size=0.2,
    random_state=42,
    stratify=y_train_full
)

y_train_split_oh = tf.keras.utils.to_categorical(y_train_split, number_of_classes)
y_val_split_oh   = tf.keras.utils.to_categorical(y_val_split, number_of_classes)

X_train_flat = X_train_split.reshape(X_train_split.shape[0], -1)
X_val_flat   = X_val_split.reshape(X_val_split.shape[0], -1)
X_test_flat  = X_test_full.reshape(X_test_full.shape[0], -1)

scaler = StandardScaler()
X_train_std = scaler.fit_transform(X_train_flat)
X_val_std   = scaler.transform(X_val_flat)
X_test_std  = scaler.transform(X_test_flat)

print("X_train_flat:", X_train_flat.shape)
print("X_val_flat  :", X_val_flat.shape)
print("X_test_flat :", X_test_flat.shape)

print("\nApplying PCA for dimensionality reduction...")
pca = PCA(n_components=100, random_state=42)
X_train_pca = pca.fit_transform(X_train_std)
X_val_pca = pca.transform(X_val_std)
X_test_pca = pca.transform(X_test_std)

print(f"Explained variance ratio: {pca.explained_variance_ratio_.sum():.4f}")

Train images: (88800, 28, 28, 1) Train labels: (88800,)
Test images : (14800, 28, 28, 1) Test labels : (14800,)
X_train_flat: (71040, 784)
X_val_flat  : (17760, 784)
X_test_flat : (14800, 784)

Applying PCA for dimensionality reduction...
Explained variance ratio: 0.8305


In [ ]:
# KNN
print("\nRunning KNN...")

knn = KNeighborsClassifier(n_neighbors=3)
knn.fit(X_train_pca, y_train_split)

knn_preds = knn.predict(X_val_pca)
knn_acc = np.mean(knn_preds == y_val_split)

print("KNN Validation Accuracy:", knn_acc)

knn_test_preds = knn.predict(X_test_pca)
knn_test_acc = np.mean(knn_test_preds == y_test_full)
print("KNN Test Accuracy:", knn_test_acc)

print("\nKNN Classification Report (Test Set):")
print(classification_report(y_test_full, knn_test_preds, zero_division=0))


Running KNN...
KNN Validation Accuracy: 0.8409346846846847
KNN Test Accuracy: 0.8260135135135135

KNN Classification Report (Test Set):
              precision    recall  f1-score   support

           1       0.77      0.85      0.80       800
           2       0.87      0.85      0.86       800
           3       0.83      0.92      0.87       800
           4       0.89      0.82      0.85       800
           5       0.87      0.88      0.87       800
           6       0.88      0.83      0.85       800
           7       0.80      0.62      0.70       800
           8       0.83      0.85      0.84       800
           9       0.68      0.69      0.68       800
          10       0.92      0.85      0.89       800
          11       0.91      0.82      0.86       800
          12       0.67      0.69      0.68       800
          13       0.96      0.92      0.94       800
          14       0.89      0.86      0.88       800
          15       0.83      0.96      0.89       80

In [ ]:
# SVM
print("\nRunning SVM...")

svm = LinearSVC(
    max_iter=10000,
    dual=False,
    tol=1e-4,
    C=0.1,
    random_state=42,
    verbose=1
)
svm.fit(X_train_pca, y_train_split)

svm_preds = svm.predict(X_val_pca)
svm_acc = np.mean(svm_preds == y_val_split)

print("SVM Validation Accuracy:", svm_acc)

svm_test_preds = svm.predict(X_test_pca)
svm_test_acc = np.mean(svm_test_preds == y_test_full)
print("SVM Test Accuracy:", svm_test_acc)
print("\nSVM Classification Report (Test Set):")
print(classification_report(y_test_full, svm_test_preds, zero_division=0))


Running SVM...
[LibLinear]SVM Validation Accuracy: 0.6886261261261262
SVM Test Accuracy: 0.6801351351351351

SVM Classification Report (Test Set):
              precision    recall  f1-score   support

           1       0.64      0.51      0.57       800
           2       0.73      0.74      0.73       800
           3       0.76      0.81      0.78       800
           4       0.75      0.57      0.65       800
           5       0.78      0.73      0.75       800
           6       0.80      0.74      0.77       800
           7       0.68      0.33      0.44       800
           8       0.71      0.67      0.69       800
           9       0.65      0.63      0.64       800
          10       0.79      0.77      0.78       800
          11       0.70      0.64      0.67       800
          12       0.63      0.57      0.60       800
          13       0.86      0.91      0.89       800
          14       0.69      0.63      0.66       800
          15       0.79      0.89      0.

In [ ]:
# K means clustering
print("\nRUNNING K-MEANS")

pca_kmeans = PCA(n_components=50, random_state=42)
X_train_pca_km = pca_kmeans.fit_transform(X_train_std)
X_val_pca_km = pca_kmeans.transform(X_val_std)

kmeans = MiniBatchKMeans(
    n_clusters=number_of_classes,
    batch_size=2048,
    max_iter=100,
    n_init=3,
    random_state=42,
    verbose=1
)

kmeans.fit(X_train_pca_km)
val_clusters = kmeans.predict(X_val_pca_km)

print("K-means ARI Score (Val):", adjusted_rand_score(y_val_split, val_clusters))

X_test_pca_km = pca_kmeans.transform(X_test_std)
test_clusters = kmeans.predict(X_test_pca_km)
print("K-means ARI Score (Test):", adjusted_rand_score(y_test_full, test_clusters))

print("Example cluster assignments:", val_clusters[:20])


RUNNING K-MEANS
Init 1/3 with method k-means++
Inertia for init 1/3: 3036219.25
Init 2/3 with method k-means++
Inertia for init 2/3: 3030401.5
Init 3/3 with method k-means++
Inertia for init 3/3: 2959837.75
Minibatch step 1/3468: mean batch inertia: 394.33349609375
Minibatch step 2/3468: mean batch inertia: 372.54986572265625, ewa inertia: 372.54986572265625
Minibatch step 3/3468: mean batch inertia: 307.0298156738281, ewa inertia: 368.7721862840222
Minibatch step 4/3468: mean batch inertia: 295.93499755859375, ewa inertia: 364.57262370720946
Minibatch step 5/3468: mean batch inertia: 363.76788330078125, ewa inertia: 364.52622491348853
Minibatch step 6/3468: mean batch inertia: 306.8102111816406, ewa inertia: 361.19850159532507
Minibatch step 7/3468: mean batch inertia: 296.0487060546875, ewa inertia: 357.44216986386783
Minibatch step 8/3468: mean batch inertia: 347.9836730957031, ewa inertia: 356.896822771873
Minibatch step 9/3468: mean batch inertia: 290.5122375488281, ewa inertia: 